In [ ]:
# Cell 0: Setup — load model, tokenizer, MBE utilities
import torch
import numpy as np
import matplotlib.pyplot as plt
from transformers import AutoModelForCausalLM, AutoTokenizer
from src.mbe import mbe_reverse_gram, OnlineMBE

MODEL_NAME = "Qwen/Qwen3-0.6B"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
dtype = torch.bfloat16 if device == "cuda" else torch.float32

model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=dtype).to(device)
model.eval()
print(f"Model loaded on {device}, {sum(p.numel() for p in model.parameters())/1e6:.0f}M params")
print(f"Layers: {model.config.num_hidden_layers}, Hidden dim: {model.config.hidden_size}")

In [ ]:
# Cell 1: Generate completions and collect hidden states via a single forward pass

prompts = [
    "What is 24 * 17 + 3? Think step by step.",
    "A store sells apples for $2 each and oranges for $3 each. If John buys 5 apples and 4 oranges, how much does he spend in total?",
    "If a train travels at 60 mph for 2.5 hours, how far does it go?",
]

rollouts = []  # will store per-rollout data

for i, prompt_text in enumerate(prompts):
    messages = [{"role": "user", "content": prompt_text}]
    chat_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(chat_text, return_tensors="pt").to(device)
    prompt_len = inputs["input_ids"].shape[1]

    with torch.no_grad():
        gen_out = model.generate(
            **inputs, max_new_tokens=512, do_sample=True, temperature=0.7,
        )

    gen_ids = gen_out[0]
    completion_ids = gen_ids[prompt_len:]
    completion_text = tokenizer.decode(completion_ids, skip_special_tokens=True)

    # Single forward pass to get all hidden states at once
    with torch.no_grad():
        full_out = model(gen_ids.unsqueeze(0), output_hidden_states=True, use_cache=False)
    # hidden_states[layer]: (1, seq_len, D) — skip layer 0 (embedding)
    hs = [h[0, prompt_len:, :].float().cpu() for h in full_out.hidden_states[1:]]

    rollouts.append({
        "prompt": prompt_text,
        "completion": completion_text,
        "prompt_len": prompt_len,
        "comp_len": len(completion_ids),
        "hidden_states": hs,  # list of (T_comp, D) per layer
        "tokens": [tokenizer.decode([tid]) for tid in completion_ids.cpu()],
    })

    print(f"\n--- Prompt {i} ({len(completion_ids)} comp tokens) ---")
    print(f"Q: {prompt_text[:100]}")
    print(f"A: {completion_text[:300]}")

n_layers = len(rollouts[0]["hidden_states"])
D = rollouts[0]["hidden_states"][0].shape[1]
print(f"\n{n_layers} layers, hidden_dim={D}")

In [ ]:
# Cell 2: Cumulative MBE — how MBE grows as the completion gets longer
# For each prefix length t, compute MBE(hidden[0:t])

def cumulative_mbe(hidden, min_tokens=2):
    """Compute MBE on prefixes hidden[0:t] for t = min_tokens..T.
    Args: hidden (T, D)
    Returns: list of (t, mbe_value) pairs
    """
    T = hidden.shape[0]
    results = []
    for t in range(min_tokens, T + 1):
        h = hidden[:t].unsqueeze(0)  # (1, t, D)
        mbe_val = mbe_reverse_gram(h).item()
        results.append((t, mbe_val))
    return results

# Compute for last layer across all rollouts
layer_idx = -1  # last layer

fig, axes = plt.subplots(1, len(rollouts), figsize=(6 * len(rollouts), 4), squeeze=False)
for ri, roll in enumerate(rollouts):
    h = roll["hidden_states"][layer_idx]
    cum = cumulative_mbe(h)
    ts, mbes = zip(*cum)
    axes[0][ri].plot(ts, mbes, linewidth=1.5)
    axes[0][ri].set_xlabel("Completion length (tokens)")
    axes[0][ri].set_ylabel("Cumulative MBE")
    axes[0][ri].set_title(f"Rollout {ri} ({roll['comp_len']} tok)\n{roll['prompt'][:50]}...")
    axes[0][ri].grid(True, alpha=0.3)

plt.suptitle(f"Cumulative MBE (layer {layer_idx}): MBE grows as completion extends", fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Cell 3: Per-layer cumulative MBE — how MBE builds up at different depths

roll = rollouts[0]

selected_layers = [0, n_layers // 4, n_layers // 2, 3 * n_layers // 4, n_layers - 1]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Whole-completion MBE per layer
layer_mbe = []
for li in range(n_layers):
    h = roll["hidden_states"][li].unsqueeze(0)  # (1, T, D)
    mbe_val = mbe_reverse_gram(h).item()
    layer_mbe.append(mbe_val)

ax1.bar(range(n_layers), layer_mbe, color="steelblue", alpha=0.8)
ax1.set_xlabel("Layer index")
ax1.set_ylabel("Whole-completion MBE")
ax1.set_title(f"MBE per layer — Rollout 0\n{roll['prompt'][:60]}...")
ax1.grid(True, alpha=0.3, axis="y")

# Cumulative MBE at selected layers
for li in selected_layers:
    h = roll["hidden_states"][li]
    cum = cumulative_mbe(h, min_tokens=4)
    ts, mbes = zip(*cum)
    ax2.plot(ts, mbes, label=f"Layer {li}", linewidth=1.2)

ax2.set_xlabel("Completion length (tokens)")
ax2.set_ylabel("Cumulative MBE")
ax2.set_title("Cumulative MBE at different layers")
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Cell 4: Compare correct vs incorrect rollouts — cumulative MBE dynamics

import re

test_prompt = "What is 15 * 13 + 7?"
gold_answer = str(15 * 13 + 7)  # "202"

messages = [{"role": "user", "content": test_prompt}]
chat_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(chat_text, return_tensors="pt").to(device)
prompt_len = inputs["input_ids"].shape[1]

n_rollouts = 8
correct_trajs = []
incorrect_trajs = []

for k in range(n_rollouts):
    with torch.no_grad():
        gen_out = model.generate(**inputs, max_new_tokens=256, do_sample=True, temperature=0.9)

    gen_ids = gen_out[0]
    comp_text = tokenizer.decode(gen_ids[prompt_len:], skip_special_tokens=True)

    numbers = re.findall(r"-?[\d,]+\.?\d*", comp_text)
    predicted = numbers[-1].replace(",", "") if numbers else ""
    is_correct = predicted == gold_answer

    with torch.no_grad():
        full_out = model(gen_ids.unsqueeze(0), output_hidden_states=True, use_cache=False)
    h = full_out.hidden_states[-1][0, prompt_len:, :].float().cpu()
    cum = cumulative_mbe(h)

    label = f"{'OK' if is_correct else 'WRONG'} (ans={predicted})"
    if is_correct:
        correct_trajs.append((cum, label))
    else:
        incorrect_trajs.append((cum, label))

print(f"Correct: {len(correct_trajs)}/{n_rollouts}, Incorrect: {len(incorrect_trajs)}/{n_rollouts}")

fig, ax = plt.subplots(figsize=(10, 5))
for cum, label in correct_trajs:
    ts, mbes = zip(*cum)
    ax.plot(ts, mbes, color="green", alpha=0.5, linewidth=1.0, label=label)
for cum, label in incorrect_trajs:
    ts, mbes = zip(*cum)
    ax.plot(ts, mbes, color="red", alpha=0.5, linewidth=1.0, label=label)

handles, labels = ax.get_legend_handles_labels()
by_label = dict(zip(labels, handles))
ax.legend(by_label.values(), by_label.keys(), fontsize=8)
ax.set_xlabel("Completion length (tokens)")
ax.set_ylabel("Cumulative MBE (last layer)")
ax.set_title(f"Cumulative MBE: correct vs incorrect\nQ: {test_prompt} → {gold_answer}")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Cell 5: Summary — cumulative MBE at key completion percentages

print(f"{'Rollout':<10} {'Tokens':<8} {'MBE@25%':>8} {'MBE@50%':>8} {'MBE@75%':>8} {'MBE@100%':>9}")
print("-" * 55)

for ri, roll in enumerate(rollouts):
    h = roll["hidden_states"][-1]
    T = h.shape[0]
    checkpoints = [max(2, int(T * f)) for f in [0.25, 0.5, 0.75, 1.0]]

    mbe_at = []
    for t in checkpoints:
        t = min(t, T)
        val = mbe_reverse_gram(h[:t].unsqueeze(0)).item()
        mbe_at.append(val)

    print(f"{ri:<10} {T:<8} {mbe_at[0]:>8.4f} {mbe_at[1]:>8.4f} {mbe_at[2]:>8.4f} {mbe_at[3]:>9.4f}")